In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('drug200.csv')
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,DrugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,DrugY


In [3]:
df.describe()

,Age,Na_to_K
count,200.000000,200.000000
mean,44.315000,16.084485
std,16.544315,7.223956
min,15.000000,6.269000
25%,31.000000,10.445500
50%,45.000000,13.936500
75%,58.000000,19.380000
max,74.000000,38.247000


In [4]:
df.isna().sum()

,0
Age,0
Sex,0
BP,0
Cholesterol,0
Na_to_K,0
Drug,0


In [5]:
df.duplicated().sum()

np.int64(0)

In [7]:
X = df.drop("Drug", axis=1)
y = df["Drug"]

In [8]:
cat_columns = ["Sex", "BP", "Cholesterol"]
le_dict = {}

for col in cat_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

In [9]:
le_drug = LabelEncoder()
y = le_drug.fit_transform(y)

print("Drug classes:", le_drug.classes_)

Drug classes: ['DrugY' 'drugA' 'drugB' 'drugC' 'drugX']


In [10]:
# Convert to numpy
X = X.values.astype(np.float32)
y = y.astype(np.int64)

# Scale numerical feature(s) — mainly Na_to_K
scaler = StandardScaler()
X[:, -1] = scaler.fit_transform(X[:, [-1]]).ravel()

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# 2. PyTorch Dataset
class DrugDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X) # Creating tensors from numpy array
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = DrugDataset(X_train, y_train)
val_ds   = DrugDataset(X_val, y_val)
# DataLoder splits the train dataset into 32 mini batches and the test dataset into 64
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

In [14]:
# Simple feed-forward NN for 5-class drug prediction.
class DrugClassifier(nn.Module):
    def __init__(self, input_dim=5, hidden=[64, 32], num_classes=5):
        super().__init__()
        layers = []

        prev = input_dim
        # Construct the hidden part of the network layer by layer.
        # Each block contains only a linear transformation followed by ReLU activation.
        for h in hidden:
            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU()
            ])
            prev = h
        # It adds one final fully-connected (dense) layer to the layers list
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = DrugClassifier(input_dim=X.shape[1], hidden=[128, 64, 32])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)

DrugClassifier(
  (net): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=5, bias=True)
  )
)


In [16]:
# 4. Training Loop
# ────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss() # loss function
optimizer = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4) # enhanced version of the adam optimizer
# schedualer lowers the learning rate when model stops impoving.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

def train_epoch():
    model.train()
    total_loss = 0
    correct = 0
    # loops through all mini-batches of training data
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb) # forward pass
        loss = criterion(logits, yb) # cross-entropy loss
        loss.backward() # compute gradients of the loss with respect to every weight in the model.
        optimizer.step() # update teh weights

        total_loss += loss.item() * Xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()

    return total_loss / len(train_ds), correct / len(train_ds)


def evaluate():
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * Xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()

    return total_loss / len(val_ds), correct / len(val_ds)

In [17]:
best_val_acc = 0
best_epoch = 0

for epoch in range(120):
    train_loss, train_acc = train_epoch()
    val_loss, val_acc = evaluate()

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(model.state_dict(), "best_drug_model.pt")
        print(f"↑ New best @ epoch {epoch:3d}")

    if epoch % 10 == 0 or epoch < 15:
        print(f"[{epoch:3d}]  train loss: {train_loss:.4f}  acc: {train_acc:.4f}  |  "
              f"val loss: {val_loss:.4f}  acc: {val_acc:.4f}")

print(f"\nBest validation accuracy: {best_val_acc:.4f}  (epoch {best_epoch})")

↑ New best @ epoch   0
[  0]  train loss: 1.5812  acc: 0.3187  |  val loss: 1.6730  acc: 0.3750
[  1]  train loss: 1.4100  acc: 0.4750  |  val loss: 1.5325  acc: 0.3750
[  2]  train loss: 1.4058  acc: 0.4750  |  val loss: 1.5549  acc: 0.3750
↑ New best @ epoch   3
[  3]  train loss: 1.3588  acc: 0.4750  |  val loss: 1.5521  acc: 0.4250
[  4]  train loss: 1.3755  acc: 0.4813  |  val loss: 1.5525  acc: 0.3750
[  5]  train loss: 1.3564  acc: 0.4750  |  val loss: 1.5269  acc: 0.3750
[  6]  train loss: 1.3324  acc: 0.4750  |  val loss: 1.4748  acc: 0.4000
[  7]  train loss: 1.3323  acc: 0.5000  |  val loss: 1.4838  acc: 0.4250
[  8]  train loss: 1.2902  acc: 0.4875  |  val loss: 1.4640  acc: 0.4250
↑ New best @ epoch   9
[  9]  train loss: 1.2736  acc: 0.5312  |  val loss: 1.4261  acc: 0.4500
↑ New best @ epoch  10
[ 10]  train loss: 1.2385  acc: 0.5687  |  val loss: 1.4069  acc: 0.5000
↑ New best @ epoch  11
[ 11]  train loss: 1.2025  acc: 0.6125  |  val loss: 1.3220  acc: 0.5250
↑ New bes